<a href="https://colab.research.google.com/github/amosagekouassi-source/DI-Bootcamp/blob/master/Evaluating_LLMs_Exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Exercises XP : Evaluating LLMs for Summarization



## What you will learn
- Hands-on evaluation for summarization: accuracy vs. ROUGE.
- Strengths/weaknesses of metrics and model size comparisons.
- Using Hugging Face `transformers` + `evaluate` for quick experiments.
- Data loading, sampling, preprocessing, and debugging model outputs.

**Create**: evaluation scripts, comparison tables, custom metrics, and short analyses.


In [ ]:
# Part I. Setup (run once per runtime)
# Install minimal deps with explicit flags and upgrade to avoid conflicts
!pip install -q --upgrade pip
!pip install -q --upgrade evaluate rouge_score datasets transformers accelerate nltk

import nltk
try:
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    nltk.download('punkt')
    nltk.download('punkt_tab')

import evaluate
print(f"Setup complete. Using 'evaluate' version: {evaluate.__version__}")


### Part II. Dataset loading and exploration
Preferred dataset: [abisee/cnn_dailymail](https://huggingface.co/datasets/abisee/cnn_dailymail) (map `article` -> `prompt_text`, `highlights` -> `prompt_title`).
- If you have local train/test CSVs with `prompt_text` / `prompt_title`, set the paths below.
- Otherwise, we will auto-sample a small slice from the HF dataset to keep things light.
- Show a couple of rows for a sanity check.
If HF download fails, a tiny fallback sample is used.


In [ ]:

import pandas as pd
from datasets import load_dataset

# Point to your data; leave empty to use the HF cnn_dailymail sample or fallback
train_path = ''  # e.g., '/content/train.csv'
test_path = ''   # e.g., '/content/test.csv'

fallback = pd.DataFrame([
    {
        'prompt_text': 'The cat sat on the mat and purred loudly while the sun set.',
        'prompt_title': 'Cat rests on mat at sunset'
    },
    {
        'prompt_text': 'Scientists discovered water on the moon, opening new research paths.',
        'prompt_title': 'Water found on the moon'
    },
    {
        'prompt_text': 'The local team won the championship after a dramatic final match.',
        'prompt_title': 'Local team clinches title'
    },
])

def load_and_sample(path, split_name, n):
    if path:
        df = pd.read_csv(path)
    else:
        try:
            hf_split = f"{split_name}[:{max(n, 3)}]"
            ds = load_dataset('abisee/cnn_dailymail', '3.0.0', split=hf_split)
            df = ds.to_pandas()[['article', 'highlights']].rename(columns={'article': 'prompt_text', 'highlights': 'prompt_title'})
        except Exception as exc:
            print(f"HF load failed ({exc}); using tiny fallback sample.")
            df = fallback.copy()
    return df.sample(min(n, len(df)), random_state=42).reset_index(drop=True)

train_df = load_and_sample(train_path, 'train', 100)
test_df = load_and_sample(test_path, 'test', 50)

display(train_df.head(2))



### Part III. Summarization with T5 (implement)
Tasks:
- Write `batch_generator` to yield mini-batches.
- Write `summarize_with_t5` using `t5-small` (or swap sizes) with GPU if available.
- Prefix inputs with "summarize: " and decode with `skip_special_tokens=True`.
- Clear CUDA cache between batches (`torch.cuda.empty_cache()`) and gc.collect().


In [ ]:
import torch, gc
from transformers import AutoTokenizer, T5ForConditionalGeneration
from typing import Iterable, List
import pandas as pd

def batch_generator(items: List[str], batch_size: int):
    for i in range(0, len(items), batch_size):
        yield items[i : i + batch_size]

def summarize_with_t5(texts: List[str], model_name: str = 't5-small', batch_size: int = 4, max_new_tokens: int = 32):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = T5ForConditionalGeneration.from_pretrained(model_name).to(device)

    summaries = []
    prefix = "summarize: "

    for batch in batch_generator(texts, batch_size):
        inputs = [prefix + text for text in batch]
        tokens = tokenizer(inputs, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)

        with torch.no_grad():
            output_tokens = model.generate(**tokens, max_new_tokens=max_new_tokens)

        batch_preds = [tokenizer.decode(g, skip_special_tokens=True) for g in output_tokens]
        summaries.extend(batch_preds)

        del tokens, output_tokens
        torch.cuda.empty_cache()
        gc.collect()

    return summaries

RUN_T5 = True
if RUN_T5:
    train_summaries_t5 = summarize_with_t5(train_df['prompt_text'].tolist(), model_name='t5-small', batch_size=4)
    results_df = pd.DataFrame({
        'prompt_text': train_df['prompt_text'],
        'reference_summary': train_df['prompt_title'],
        't5_small_summary': train_summaries_t5
    })
    display(results_df.head())
else:
    print("Skipping T5 generation for speed. Set RUN_T5=True to execute.")


### Part IV. Accuracy evaluation (toy, likely near zero)
Implement a naive accuracy that checks exact string match between generated and reference summaries.
Discuss why this is harsh for free-form text (almost always zero).


In [ ]:

from typing import List

def compute_accuracy(preds: List[str], refs: List[str]) -> float:
    matches = sum(1 for p, r in zip(preds, refs) if p.strip() == r.strip())
    return matches / max(len(refs), 1)

if 'train_summaries_t5' in locals():
    acc = compute_accuracy(train_summaries_t5, train_df['prompt_title'].tolist())
    print(f"Exact-match accuracy: {acc:.4f}")
else:
    print("Accuracy skipped (no predictions).")



### Part V. ROUGE metric implementation
Use `evaluate.load("rouge")` and NLTK sentence tokenizer.
Preprocess by joining sentences with newlines for better ROUGE-L.


In [ ]:
try:
    import evaluate
except ImportError:
    !pip install -q evaluate rouge_score
    import evaluate

import nltk
from nltk.tokenize import sent_tokenize
from typing import List

# Ensure nltk resources are available
try:
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    nltk.download('punkt')
    nltk.download('punkt_tab')

rouge = evaluate.load('rouge')

def normalize_text(text):
    sents = sent_tokenize(str(text).strip())
    return "\n".join(sents)

def compute_rouge_score(preds: List[str], refs: List[str]):
    norm_preds = [normalize_text(p) for p in preds]
    norm_refs = [normalize_text(r) for r in refs]
    return rouge.compute(predictions=norm_preds, references=norm_refs)

test_preds = ["alpha beta", "", "The cat sat."]
test_refs  = ["alpha beta", "reference text", "The cat sat."]
print("ROUGE sanity check:")
print(compute_rouge_score(test_preds, test_refs))


### Part VI. Understanding ROUGE scores
Experiments to run (describe your findings in a text cell):
- Exact match vs. empty prediction.
- Effect of stemming: e.g., "running" vs. "run".
- N-gram overlap: see how ROUGE-1 vs. ROUGE-2 change with partial overlap.
- Symmetry: swap preds/refs and compare.



### Part VII. Comparing small and large models
Goals:
- Generate summaries with `t5-small`, `t5-base`, and `gpt2` (TL;DR style prompt).
- Compute ROUGE for each and store per-row scores.
- Implement `compute_rouge_per_row` to add ROUGE columns to a DataFrame.
- Implement `summarize_with_gpt2` with a TL;DR: prefix and max length guard.
Use small batches and low `max_new_tokens` to keep things snappy.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch, gc

def summarize_with_gpt2(texts: List[str], model_name: str = 'gpt2', batch_size: int = 2, max_new_tokens: int = 32):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

    summaries = []
    for batch in batch_generator(texts, batch_size):
        inputs = [text[:500] + "\n\nTL;DR:" for text in batch]
        tokens = tokenizer(inputs, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)

        with torch.no_grad():
            output_tokens = model.generate(**tokens, max_new_tokens=max_new_tokens, pad_token_id=tokenizer.eos_token_id)

        for i, output in enumerate(output_tokens):
            decoded = tokenizer.decode(output, skip_special_tokens=True)
            summary = decoded.split("TL;DR:")[-1].strip()
            summaries.append(summary)

        torch.cuda.empty_cache()
        gc.collect()
    return summaries

def compute_rouge_per_row(df: pd.DataFrame, pred_col: str, ref_col: str = 'reference_summary'):
    scores = []
    for _, row in df.iterrows():
        score = compute_rouge_score([row[pred_col]], [row[ref_col]])
        scores.append(score['rougeL'])
    return scores

RUN_COMPARE = True
if RUN_COMPARE and 'train_summaries_t5' in locals():
    # Calculate T5 scores
    results_df['t5_rougeL'] = compute_rouge_per_row(results_df, 't5_small_summary')

    # Generate and calculate GPT-2 scores for a small subset to save time
    print("Generating GPT-2 summaries for comparison...")
    gpt2_sample_texts = train_df['prompt_text'].tolist()[:10]
    gpt2_summaries = summarize_with_gpt2(gpt2_sample_texts)

    # For the sake of the dataframe, we'll pad the rest with empty strings
    full_gpt2_list = gpt2_summaries + [""] * (len(results_df) - len(gpt2_summaries))
    results_df['gpt2_summary'] = full_gpt2_list
    results_df['gpt2_rougeL'] = compute_rouge_per_row(results_df, 'gpt2_summary')

    print("Scores for T5 and GPT-2 (partial) computed.")


### Part VIII. Comparing all models
Implement:
- `compare_models` to aggregate average ROUGE across models.
- `compare_models_summaries` to show side-by-side summaries.
Present the tables and discuss which model wins and why.


In [ ]:
import pandas as pd

def compare_models(rouge_dict):
    """rouge_dict: {'model_name': {'rouge1': 0.1, ...}}"""
    # Filter to only keep relevant ROUGE keys for the table
    cleaned_dict = {}
    for model, scores in rouge_dict.items():
        cleaned_dict[model] = {k: v for k, v in scores.items() if k in ['rouge1', 'rouge2', 'rougeL']}
    return pd.DataFrame(cleaned_dict).T

def compare_models_summaries(df: pd.DataFrame, pred_cols: list):
    cols = ['reference_summary'] + pred_cols
    # Filtering only rows where we have GPT-2 output (the first 10)
    return df[cols].head(10)

if RUN_COMPARE and 'results_df' in locals():
    # Aggregate Average Scores
    avg_t5 = compute_rouge_score(results_df['t5_small_summary'].tolist(), results_df['reference_summary'].tolist())

    # For GPT-2, we only compute on the first 10 rows where we actually generated text
    gpt2_valid = results_df[results_df['gpt2_summary'] != ""]
    avg_gpt2 = compute_rouge_score(gpt2_valid['gpt2_summary'].tolist(), gpt2_valid['reference_summary'].tolist())

    comparison_table = compare_models({
        'T5-Small': avg_t5,
        'GPT-2 (Subset)': avg_gpt2
    })

    print("--- Summary Performance Comparison ---")
    display(comparison_table)

    print("\n--- Qualitative Side-by-Side (First 5 Rows) ---")
    display(compare_models_summaries(results_df, ['t5_small_summary', 'gpt2_summary']).head(5))


## Wrap-up
- Which metrics felt most informative? Why?
- How did model size impact ROUGE and qualitative quality?
- Where did accuracy break down as a metric?
- How would you extend this to human eval or adversarial probes?
Write a short reflection here.
